In [1]:
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama "mcp[cli]<2" httpx gradio

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b

In [3]:
%%writefile weather_mcp_server.py
"""MCP server exposing weather tools backed by the free Open-Meteo API.

Install:  pip install "mcp[cli]" httpx
Run:      python weather_mcp_server.py   (stdio transport)
"""
import httpx
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather")

GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


@mcp.tool()
async def get_coordinates(city: str) -> dict:
    """Look up latitude/longitude for a city name."""
    async with httpx.AsyncClient(timeout=15) as client:
        r = await client.get(GEO_URL, params={"name": city, "count": 1})
        r.raise_for_status()
        results = r.json().get("results")
    if not results:
        return {"error": f"No match for '{city}'"}
    top = results[0]
    return {
        "name": top["name"],
        "country": top.get("country"),
        "latitude": top["latitude"],
        "longitude": top["longitude"],
    }


@mcp.tool()
async def get_forecast(latitude: float, longitude: float, days: int = 3) -> dict:
    """Get current conditions and a daily forecast (1-7 days) for coordinates."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,wind_speed_10m,precipitation",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "forecast_days": max(1, min(days, 7)),
        "timezone": "auto",
    }
    async with httpx.AsyncClient(timeout=15) as client:
        r = await client.get(FORECAST_URL, params=params)
        r.raise_for_status()
        return r.json()


if __name__ == "__main__":
    mcp.run(transport="stdio")


Writing weather_mcp_server.py


In [4]:
import sys
import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL = "qwen2.5:7b"
client = ollama.AsyncClient()

SYSTEM = ("You are a helpful weather assistant. Always use tools to get real data: "
          "first get_coordinates for the city, then get_forecast.")

def schema_of(tool):
    # mcp 1.x uses inputSchema, mcp 2.x uses input_schema
    return getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None)

async def ask(question: str, history=None, log=print) -> str:
    """Run one question through the MCP-backed agent and return the answer text."""
    server = StdioServerParameters(command=sys.executable, args=["weather_mcp_server.py"])
    async with stdio_client(server, errlog=open("mcp_server.log", "w")) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools

            tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description or "",
                    "parameters": schema_of(t),
                },
            } for t in mcp_tools]

            messages = [{"role": "system", "content": SYSTEM}]
            messages += history or []
            messages.append({"role": "user", "content": question})

            for _ in range(6):
                resp = await client.chat(model=MODEL, messages=messages, tools=tools)
                msg = resp.message
                messages.append(msg)

                if not msg.tool_calls:
                    return msg.content

                for call in msg.tool_calls:
                    name, args = call.function.name, dict(call.function.arguments)
                    log(f"[tool] {name}({args})")
                    out = await session.call_tool(name, args)
                    text = "".join(getattr(c, "text", "") for c in out.content)
                    messages.append({"role": "tool", "tool_name": name, "content": text})
            return "Stopped: too many tool rounds."

async def run(question: str):
    print(await ask(question))


In [5]:
await run("Should I bring an umbrella in Paris over the next 3 days?")

[tool] get_coordinates({'city': 'Paris'})
[tool] get_forecast({'latitude': 48.8566, 'longitude': 2.3522, 'days': 3})
Based on the forecast for Paris over the next 3 days, there is no predicted precipitation. Therefore, you likely do not need to bring an umbrella. However, it's always a good idea to keep one with you just in case unexpected weather changes. Enjoy your stay in Paris!


In [6]:
!pip install -q fastapi uvicorn

In [8]:
import asyncio, threading, time, uvicorn
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from google.colab import output

main_loop = asyncio.get_running_loop()  # the notebook loop, where ask() must run

PAGE = r"""<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Weather Agent</title>
<style>
*{box-sizing:border-box}
body{margin:0;min-height:100vh;display:flex;flex-direction:column;align-items:center;color:#e8ecff;
 font-family:system-ui,sans-serif;background:radial-gradient(circle at 50% 0%,#1d2a5c,#0a0e22 60%)}
.orbwrap{margin:28px 0 8px;position:relative;width:130px;height:130px}
.orb{position:absolute;inset:18px;border-radius:50%;
 background:radial-gradient(circle at 35% 30%,#9fd3ff,#4f7cff 55%,#2a2f8f);
 box-shadow:0 0 40px #4f7cff88;animation:breathe 4s ease-in-out infinite}
.ring{position:absolute;inset:0;border-radius:50%;opacity:0;
 background:conic-gradient(from 0deg,transparent,#7cf0ff,#b36bff,transparent 70%);
 -webkit-mask:radial-gradient(circle,transparent 58%,#000 60%);mask:radial-gradient(circle,transparent 58%,#000 60%);
 transition:opacity .4s}
.thinking .ring{opacity:1;animation:spin 1.1s linear infinite}
.thinking .orb{animation:pulse .9s ease-in-out infinite;box-shadow:0 0 70px #b36bffaa}
.listening .orb{background:radial-gradient(circle at 35% 30%,#ffd1d1,#ff4f6e 55%,#8f2a3c);
 box-shadow:0 0 60px #ff4f6e99;animation:pulse .7s ease-in-out infinite}
@keyframes spin{to{transform:rotate(360deg)}}
@keyframes breathe{50%{transform:scale(1.06)}}
@keyframes pulse{50%{transform:scale(1.14)}}
h1{font-size:20px;margin:0 0 4px;font-weight:600}
.sub{opacity:.6;font-size:13px;margin-bottom:12px;height:18px}
#log{width:min(720px,94vw);flex:1;overflow-y:auto;padding:8px;display:flex;flex-direction:column;gap:10px}
.msg{max-width:85%;padding:10px 14px;border-radius:16px;line-height:1.45;animation:in .35s ease;
 white-space:pre-wrap;backdrop-filter:blur(6px)}
.user{align-self:flex-end;background:#4f7cff;border-bottom-right-radius:4px}
.assistant{align-self:flex-start;background:#ffffff14;border:1px solid #ffffff22;border-bottom-left-radius:4px}
.chip{display:inline-block;margin:6px 6px 0 0;padding:2px 9px;font-size:11px;border-radius:99px;
 background:#7cf0ff22;border:1px solid #7cf0ff55;color:#9ff3ff}
.dots span{display:inline-block;width:8px;height:8px;margin:0 3px;border-radius:50%;background:#9fb8ff;
 animation:bounce 1s infinite}
.dots span:nth-child(2){animation-delay:.15s}.dots span:nth-child(3){animation-delay:.3s}
@keyframes bounce{50%{transform:translateY(-7px);opacity:.4}}
@keyframes in{from{opacity:0;transform:translateY(8px)}}
form{width:min(720px,94vw);display:flex;gap:8px;padding:12px 0 20px}
input{flex:1;padding:13px 16px;border-radius:99px;border:1px solid #ffffff33;background:#ffffff12;color:#fff;font-size:15px;outline:none}
input:focus{border-color:#7cf0ff}
button{border:0;border-radius:99px;padding:0 18px;font-size:15px;cursor:pointer;color:#fff;background:#4f7cff}
button.ghost{background:#ffffff1c;width:46px;padding:0}
button.on{background:#ff4f6e}
</style></head>
<body>
<div class="orbwrap" id="orbwrap"><div class="ring"></div><div class="orb"></div></div>
<h1>Weather Agent</h1><div class="sub" id="sub">Ask me about the weather anywhere</div>
<div id="log"></div>
<form id="f">
 <button type="button" class="ghost" id="mic" title="Speak">&#127908;</button>
 <input id="inp" placeholder="Ask about the weather..." autocomplete="off">
 <button type="button" class="ghost" id="spk" title="Read answers aloud">&#128266;</button>
 <button type="submit">Send</button>
</form>
<script>
var log=document.getElementById('log'),inp=document.getElementById('inp'),wrap=document.getElementById('orbwrap'),
    sub=document.getElementById('sub'),mic=document.getElementById('mic'),spk=document.getElementById('spk');
var chatHistory=[],speakOn=false,busy=false;
function state(s,t){wrap.className='orbwrap '+s;sub.textContent=t||'';}
function add(role,text,tools){
  var d=document.createElement('div');d.className='msg '+role;d.textContent=text;
  (tools||[]).forEach(function(t){var c=document.createElement('span');c.className='chip';c.textContent=t;d.appendChild(document.createElement('br'));d.appendChild(c);});
  log.appendChild(d);log.scrollTop=log.scrollHeight;return d;}
async function send(text){
  text=text.trim();if(!text||busy)return;busy=true;inp.value='';
  add('user',text);
  var t=add('assistant','');t.innerHTML='<span class="dots"><span></span><span></span><span></span></span>';
  state('thinking','Calling weather tools...');
  try{
    var r=await fetch('chat',{method:'POST',headers:{'Content-Type':'application/json'},
      body:JSON.stringify({message:text,history:chatHistory})});
    var d=await r.json();t.remove();
    if(!r.ok||d.answer===undefined){add('assistant','Server error: '+JSON.stringify(d.detail||d));}
    else{
      add('assistant',d.answer,d.tools);
      chatHistory.push({role:'user',content:text},{role:'assistant',content:d.answer});
      if(speakOn&&window.speechSynthesis){speechSynthesis.speak(new SpeechSynthesisUtterance(d.answer));}
    }
  }catch(e){t.remove();add('assistant','Something went wrong: '+e);}
  busy=false;state('','Ask me about the weather anywhere');}
document.getElementById('f').onsubmit=function(e){e.preventDefault();send(inp.value);};
spk.onclick=function(){speakOn=!speakOn;spk.classList.toggle('on',speakOn);if(!speakOn)speechSynthesis.cancel();};
var SR=window.SpeechRecognition||window.webkitSpeechRecognition;
if(!SR){mic.style.display='none';}else{
  var rec=new SR();rec.lang='en-US';rec.interimResults=false;
  rec.onstart=function(){state('listening','Listening...');mic.classList.add('on');};
  rec.onend=function(){mic.classList.remove('on');if(!busy)state('','Ask me about the weather anywhere');};
  rec.onresult=function(e){send(e.results[0][0].transcript);};
  rec.onerror=function(e){sub.textContent='Mic error: '+e.error;};
  mic.onclick=function(){try{rec.start();}catch(x){}};}
</script></body></html>"""

app = FastAPI()

class ChatIn(BaseModel):
    message: str
    history: list = []

@app.get("/", response_class=HTMLResponse)
async def index():
    return PAGE

@app.post("/chat")
async def chat(body: ChatIn):
    tools = []
    try:
        # run ask() on the notebook's event loop (where the Ollama client lives)
        fut = asyncio.run_coroutine_threadsafe(
            ask(body.message, body.history, log=tools.append), main_loop)
        answer = await asyncio.wrap_future(fut)
    except BaseException as e:
        subs = getattr(e, "exceptions", [e])
        answer = "Error: " + "; ".join(repr(s) for s in subs)
    return {"answer": answer, "tools": tools}

# restart cleanly if you rerun this cell
if "ui_server" in globals():
    ui_server.should_exit = True
    time.sleep(1.5)
ui_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=ui_server.run, daemon=True).start()
time.sleep(2)

output.serve_kernel_port_as_window(8000)  # click the link it prints to open the UI

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>